# Jam Transformer — Slakh2100-redux MIDI 스트리밍 다운로드

Zenodo의 **Slakh2100-redux**(중복 제거, **1710곡**) FLAC tarball(**104.3 GB**)을
**스트리밍**하면서 `*.mid` + `metadata.yaml`만 추출합니다. 오디오는 통과만 하고
디스크엔 **~150 MB**(MIDI/메타데이터)만 남으며, 다 받으면 **자동으로 tar로 묶어
Google Drive에 업로드**합니다 — `런타임 > 모두 실행` 후 자고 일어나면 끝나 있습니다.

| 단계 | 내용 |
|---|---|
| 설정 | 옵션(omitted 포함 여부, Drive 경로) |
| 0 | (선택) 속도 프로브 2분 → ETA 확인 |
| 1 | Drive 마운트 |
| 2 | 스트리밍 추출 → **끝나면 자동으로 Drive 저장** |
| 3 | 이후 사용법 |

> ⚠️ **단일 gzip 스트림이라 재개가 불가능합니다.** 한 세션 안에 끝나야 합니다.
> (측정 35 MB/s 기준 약 50분 → 12h 세션에 충분)
>
> ℹ️ 추출은 휘발성 `/content`에 했다가 **마지막에 자동으로** Drive에 단일 tar로
> 복사합니다 (작은 파일 수천 개를 Drive FUSE에 직접 쓰는 병목 회피).
>
> **런타임:** GPU 불필요. `런타임 유형 변경 > CPU` 로 두면 할당량을 아낍니다.


## 설정

In [ ]:
# ============================================================
#  여기만 수정하세요
# ============================================================
INCLUDE_OMITTED = False   # True면 omitted(중복) 포함 → 풀 2100. 기본 False = dedup 1710
FORCE           = True    # 추출 디렉토리를 비우고 새로 시작

# 최종 결과(MIDI tar)를 저장할 Google Drive 경로
DRIVE_OUT = '/content/drive/MyDrive/slakh2100_redux_midi.tar.gz'

URL      = "https://zenodo.org/records/4599666/files/slakh2100_flac_redux.tar.gz?download=1"
TOTAL_GB = 104.3
DEST     = '/content/slakh2100_redux'
# ============================================================
print("INCLUDE_OMITTED =", INCLUDE_OMITTED, "| FORCE =", FORCE)
print("DRIVE_OUT       =", DRIVE_OUT)

## 0. (선택) 속도 프로브 — 2분만 받아보고 ETA 확인

**무인 실행(모두 실행) 시에는 이 셀을 건너뛰어도 됩니다** (불필요하게 4 GB를 더
받음). 속도가 궁금할 때만 단독 실행하세요. 5 MB/s 이상이면 한 세션으로 충분합니다.


In [ ]:
import urllib.request, time

PROBE_SEC = 120
req = urllib.request.Request(URL, headers={"User-Agent": "colab-slakh"})
read = 0; t0 = time.time()
with urllib.request.urlopen(req) as r:
    while time.time() - t0 < PROBE_SEC:
        b = r.read(1 << 20)
        if not b:
            break
        read += len(b)
mbps  = read / 1e6 / max(time.time() - t0, 1e-9)
eta_h = TOTAL_GB * 1000 / max(mbps, 1e-9) / 3600
print(f"측정: {read/1e6:.0f} MB in {time.time()-t0:.0f}s  ->  {mbps:.1f} MB/s")
print(f"104.3 GB ETA: {eta_h:.1f} h ({eta_h*60:.0f} min)")
print("→ 한 세션 스트리밍 OK" if eta_h < 10
      else "→ 너무 느림: Range-이어받기 방식 고려")

## 1. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print("Drive 마운트 완료")

## 2. 스트리밍 추출 → 끝나면 자동으로 Drive 저장

104 GB를 흘려보내며 `*.mid`/`metadata.yaml`만 추출하고, **완료 즉시** tar로 묶어
`DRIVE_OUT` 으로 복사합니다. 진행 표시(파일수·누적 GB·MB/s·ETA)는 200파일마다.


In [ ]:
import urllib.request, tarfile, time, shutil, os, subprocess
from pathlib import Path

dest = Path(DEST)
if FORCE and dest.exists():
    shutil.rmtree(dest)
dest.mkdir(parents=True, exist_ok=True)


class _Counter:
    """Count bytes streamed for progress reporting."""
    def __init__(self, f):
        self.f, self.n = f, 0

    def read(self, sz=-1):
        b = self.f.read(sz); self.n += len(b); return b


def _wanted(name):
    base = name.rsplit("/", 1)[-1]
    if not (name.endswith(".mid") or base == "metadata.yaml"):
        return False
    if not INCLUDE_OMITTED and "/omitted/" in name:
        return False
    if name.startswith(("/", "\\")) or ".." in name.split("/"):
        return False
    return True


# ---- stream + extract ----------------------------------------------------
req = urllib.request.Request(URL, headers={"User-Agent": "colab-slakh"})
n = 0; t0 = time.time()
print(f"스트리밍 시작 — omitted(중복): {'포함' if INCLUDE_OMITTED else '제외'}")
with urllib.request.urlopen(req) as resp:
    r = _Counter(resp)
    with tarfile.open(fileobj=r, mode="r|gz") as tf:
        for m in tf:
            if not m.isfile() or not _wanted(m.name):
                continue
            src = tf.extractfile(m)
            if src is None:
                continue
            p = dest / m.name
            p.parent.mkdir(parents=True, exist_ok=True)
            with src, open(p, "wb") as o:
                shutil.copyfileobj(src, o)
            n += 1
            if n % 200 == 0:
                dt   = time.time() - t0
                gb   = r.n / 1e9
                rate = gb / max(dt, 1e-9)
                eta  = (TOTAL_GB - gb) / max(rate, 1e-9) / 60
                print(f"  {n:5d} files | {gb:5.1f}/{TOTAL_GB} GB | "
                      f"{rate*1000:5.1f} MB/s | {dt/60:5.1f} min | ETA {eta:4.0f} min")

tracks = sum(1 for _ in dest.rglob("all_src.mid"))
print(f"\n추출 완료: {n} files, {tracks} tracks, {r.n/1e9:.1f} GB streamed, "
      f"{(time.time()-t0)/60:.1f} min")

# ---- auto pack + upload to Drive -----------------------------------------
print("\ntar로 묶어 Drive에 업로드 중...")
local_tar = '/content/slakh2100_redux_midi.tar.gz'
subprocess.run(['tar', 'czf', local_tar, '-C', '/content', 'slakh2100_redux'],
               check=True)
os.makedirs(os.path.dirname(DRIVE_OUT), exist_ok=True)
shutil.copy(local_tar, DRIVE_OUT)
print(f"✅ 저장 완료: {DRIVE_OUT}  ({os.path.getsize(DRIVE_OUT)/1e6:.1f} MB, "
      f"{tracks} tracks)")

## 3. 이후 사용법 (로컬 / 학습 서버)

```bash
# Drive에서 받은 tar 풀기 (수백 MB)
tar -xzf slakh2100_redux_midi.tar.gz -C data/raw
# → data/raw/slakh2100_redux/{train,validation,test}/Track*/{all_src.mid, MIDI/, metadata.yaml}

# fixed 코드(instrument 모드 fallback 수정 반영)로 재전처리
python scripts/prepare_data.py \
  --slakh_dir data/raw/slakh2100_redux --out_dir data/processed
```

> 추출물에 `all_src.mid` + `MIDI/SXX.mid` + `metadata.yaml`이 모두 포함되어
> `--slakh_melody instrument`(GT 악기명) 방식이 그대로 동작합니다.
